# State与Memory：恢复、冲突和过滤

实验共用[SQLite参考实现](../05-code/state-memory-python/README.md)。时间100/110/120是人工教学时钟，不是真实用户记录；策略是确定性规则，不能把结果称为LLM能力提升。

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "10-Knowledge").is_dir())
sys.path.insert(0, str(ROOT / "10-Knowledge/07-state-and-memory/05-code/state-memory-python/src"))
from state_memory import CheckpointStore, MemoryStore, VersionConflict, MemoryConflict
import tempfile
folder = tempfile.TemporaryDirectory()
db = str(Path(folder.name) / "learning.db")

## 独立连接恢复同一份状态

版本0代表不存在。第一个写入保存版本1；另一个连接可加载已提交内容，但携带旧版本0写入会被拒绝。

In [2]:
a, b = CheckpointStore(db), CheckpointStore(db)
v = a.save("run-1", {"step":1, "document_ids":["doc-1"]})
print("恢复:", b.load("run-1"))
try:
    b.save("run-1", {"step":9}, expected_version=0)
except VersionConflict as error:
    print("冲突:", type(error).__name__, str(error))
else:
    raise AssertionError("stale write should fail")
assert a.load("run-1")[1]["step"] == 1
a.close(); b.close()

恢复: (1, {'step': 1, 'document_ids': ['doc-1']})
冲突: VersionConflict expected 0, actual 1


## 同一个词，不同主体与有效期

A和B的语言偏好不同，A的预算在110到期。比较无Memory、直接取全量同名历史的最后一条、过滤后的Memory。

In [3]:
store = MemoryStore(db)
rows = [
    {"subject":"alice", "key":"language", "value":"Python", "source":"user:1", "now":100},
    {"subject":"bob", "key":"language", "value":"TypeScript", "source":"user:2", "now":101},
    {"subject":"alice", "key":"budget", "value":8000, "source":"task:1", "now":100, "ttl":10}
]
records = [store.put(**row) for row in rows]
queries = [("alice","language","Python"), ("alice","budget",None), ("bob","language","TypeScript")]
scores = {"无Memory":0, "全量同名历史":0, "按主体时间过滤":0}
for subject,key,expected in queries:
    candidates = [r for r in rows if r["key"] == key]
    raw = candidates[-1]["value"] if candidates else None
    memory = store.get(subject,key,now=120)
    filtered = memory.value if memory else None
    values = {"无Memory":None, "全量同名历史":raw, "按主体时间过滤":filtered}
    print(subject, key, "期望:", expected, "各策略:", values)
    for name,value in values.items():
        scores[name] += int(value == expected)
print("3个手工反例上的正确数量:", scores)
assert scores == {"无Memory":1, "全量同名历史":1, "按主体时间过滤":3}

alice language 期望: Python 各策略: {'无Memory': None, '全量同名历史': 'TypeScript', '按主体时间过滤': 'Python'}
alice budget 期望: None 各策略: {'无Memory': None, '全量同名历史': 8000, '按主体时间过滤': None}
bob language 期望: TypeScript 各策略: {'无Memory': None, '全量同名历史': 'TypeScript', '按主体时间过滤': 'TypeScript'}
3个手工反例上的正确数量: {'无Memory': 1, '全量同名历史': 1, '按主体时间过滤': 3}


## 明确更新、删除与防旧写入复活

偏好更新携带旧版本；删除后不仅读取不到，删除前的写入者也无法把记录重新覆盖回来。

In [4]:
old = records[0]
new = store.put("alice", "language", "Rust", "user:3", now=121, expected_version=old.version)
print("版本变化:", old.version, "->", new.version)
assert store.forget("alice", "language")
print("删除后读取:", store.get("alice", "language", now=122))
try:
    store.put("alice", "language", "Python", "stale:1", now=123, expected_version=new.version)
except MemoryConflict as error:
    print("旧写入被拒绝:", str(error))
else:
    raise AssertionError("deleted record must not be resurrected by stale writer")
assert store.retrieve("alice", "language", now=123) == []
store.close(); folder.cleanup()

版本变化: 1 -> 2
删除后读取: None
旧写入被拒绝: expected 2, actual 3


## 边界与下一步

过滤策略在这3个构造问题上全对，是因为测试专门检查主体和有效期，不意味着任何真实模型加入Memory都会提高成功率。删除测试只覆盖活动存储接口，不涵盖备份与磁盘残留。下一步按[Memory评测](../01-concepts/03-memory-evaluation.md)构造自己的写入、召回和使用错误集。